# RHCR Learned Results (Multiple Configs)

读取 `RHCR/exp_learned` 下多个 learned 配置目录，画 `throughput` 和 `time`（由 `run.log` 解析）。

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 60)


In [ ]:
# ===== 配置 =====
exp_root = Path('/home/shiqi/masterarbeit/RHCR/exp_learned')

# 方式1：自动扫描 exp_learned 下含 summary.csv 的目录
run_dirs = sorted([d for d in exp_root.rglob('*') if d.is_dir() and (d / 'summary.csv').exists()])

# 方式2：手动指定（取消注释）
# run_dirs = [
#     Path('/home/shiqi/masterarbeit/RHCR/exp_learned/rhcr_eval_runs/rhcr_compare_20260510_072748'),
# ]

assert run_dirs, f'No result dirs with summary.csv found under {exp_root}'
print('found runs:', len(run_dirs))
for d in run_dirs[:10]:
    print('-', d)
if len(run_dirs) > 10:
    print('...')

x_key = 'num_agents'
x_label = 'agent_num'
fig_w, fig_h, line_w = 5.5, 5.0, 2.0


In [ ]:
def parse_solver_times(run_log: Path):
    if not run_log.exists():
        return []
    out = []
    for ln in run_log.read_text(encoding='utf-8', errors='ignore').splitlines():
        if ':Succeed,' not in ln:
            continue
        try:
            payload = ln.split(':Succeed,', 1)[1]
            t = float(payload.split(',', 1)[0])
            out.append(t)
        except Exception:
            pass
    return out

records = []
for rd in run_dirs:
    df = pd.read_csv(rd / 'summary.csv')
    # 仅 learned
    if 'mode' in df.columns:
        df = df[df['mode'] == 'learned'].copy()
    if df.empty:
        continue

    label = rd.name
    for _, r in df.iterrows():
        if str(r.get('status', '')) != 'ok':
            continue
        run_dir = Path(str(r['run_dir']))
        times = parse_solver_times(run_dir / 'run.log')
        mean_t = float(np.mean(times)) if len(times) else np.nan
        sum_t = float(np.sum(times)) if len(times) else np.nan

        records.append({
            'config': label,
            'num_agents': int(r['num_agents']),
            'seed': int(r['seed']),
            'throughput_per_step': float(r['throughput_per_step']),
            'mean_solver_time': mean_t,
            'sum_solver_time': sum_t,
        })

all_df = pd.DataFrame(records)
assert not all_df.empty, 'No learned rows found.'
display(all_df.head(20))
print('configs:', sorted(all_df['config'].unique()))


In [ ]:
# 聚合 throughput
agg_thr = (all_df.groupby(['config', 'num_agents'])['throughput_per_step']
           .agg(['mean','std','count'])
           .reset_index())
agg_thr['sem'] = agg_thr['std'] / np.sqrt(agg_thr['count'].clip(lower=1))
agg_thr['ci95'] = 1.96 * agg_thr['sem']
display(agg_thr.sort_values(['num_agents','config']))

# 聚合 time（每次规划调用平均时间）
agg_time = (all_df.dropna(subset=['mean_solver_time'])
            .groupby(['config', 'num_agents'])['mean_solver_time']
            .agg(['mean','std','count'])
            .reset_index())
agg_time['sem'] = agg_time['std'] / np.sqrt(agg_time['count'].clip(lower=1))
agg_time['ci95'] = 1.96 * agg_time['sem']
display(agg_time.sort_values(['num_agents','config']))


In [ ]:
# 图1: throughput
plt.figure(figsize=(fig_w, fig_h))
for cfg in sorted(agg_thr['config'].unique()):
    sub = agg_thr[agg_thr['config'] == cfg].sort_values('num_agents')
    x = sub['num_agents'].tolist()
    y = sub['mean'].tolist()
    c = sub['ci95'].fillna(0).tolist()
    plt.plot(x, y, marker='o', linewidth=line_w, label=cfg)
    plt.fill_between(x, [yy-cc for yy,cc in zip(y,c)], [yy+cc for yy,cc in zip(y,c)], alpha=0.18)

plt.title('Learned Throughput Comparison')
plt.xlabel(x_label)
plt.ylabel('throughput_per_step')
xt = sorted([int(v) for v in agg_thr['num_agents'].dropna().unique().tolist()])
if xt:
    plt.xticks(xt)
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
out1 = exp_root / 'learned_throughput_compare.pdf'
plt.savefig(out1)
print('Saved:', out1)
plt.show()


In [ ]:
# 图2: time（每次规划调用平均时间）
plt.figure(figsize=(fig_w, fig_h))
for cfg in sorted(agg_time['config'].unique()):
    sub = agg_time[agg_time['config'] == cfg].sort_values('num_agents')
    x = sub['num_agents'].tolist()
    y = sub['mean'].tolist()
    c = sub['ci95'].fillna(0).tolist()
    plt.plot(x, y, marker='o', linewidth=line_w, label=cfg)
    plt.fill_between(x, [yy-cc for yy,cc in zip(y,c)], [yy+cc for yy,cc in zip(y,c)], alpha=0.18)

plt.title('Learned Compute-Time Comparison (mean per planning call)')
plt.xlabel(x_label)
plt.ylabel('time (s)')
xt = sorted([int(v) for v in agg_time['num_agents'].dropna().unique().tolist()])
if xt:
    plt.xticks(xt)
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
out2 = exp_root / 'learned_time_compare.pdf'
plt.savefig(out2)
print('Saved:', out2)
plt.show()
